In [1]:
import importlib.util
import sys
from pathlib import Path

SCHEMAS_DIR = Path("/home/hello/Projects/Statements/code/moltie/schemas")
assert SCHEMAS_DIR.exists(), f"Missing schemas dir: {SCHEMAS_DIR}"

PKG = "schemas"  # pretend package name

# Create an empty package module object for `schemas`
if PKG not in sys.modules:
    pkg_spec = importlib.util.spec_from_loader(PKG, loader=None)
    pkg_mod = importlib.util.module_from_spec(pkg_spec)
    pkg_mod.__path__ = [str(SCHEMAS_DIR)]  # mark as package
    sys.modules[PKG] = pkg_mod

def import_as_pkg(mod_name: str, file_path: Path):
    full_name = f"{PKG}.{mod_name}"
    spec = importlib.util.spec_from_file_location(full_name, str(file_path))
    assert spec and spec.loader, f"Cannot load spec for {file_path}"
    mod = importlib.util.module_from_spec(spec)
    sys.modules[full_name] = mod
    spec.loader.exec_module(mod)
    return mod

query_object  = import_as_pkg("query_object",  SCHEMAS_DIR / "query_object.py")
verdict       = import_as_pkg("verdict",       SCHEMAS_DIR / "verdict.py")
negative_exit = import_as_pkg("negative_exit", SCHEMAS_DIR / "negative_exit.py")
run_config    = import_as_pkg("run_config",    SCHEMAS_DIR / "run_config.py")

QueryObject  = query_object.QueryObject
AtomQuery    = query_object.AtomQuery
Verdict      = verdict.Verdict
Anchor       = verdict.Anchor
NegativeExit = negative_exit.NegativeExit
RunConfig    = run_config.RunConfig

print("✅ Loaded schemas as an in-memory package namespace")


✅ Loaded schemas as an in-memory package namespace


In [2]:
from pathlib import Path
import PyPDF2

appeal_dir = Path("/home/hello/Projects/Statements/code/appeals/")
appeal_name = "SAINSBURYS SUPERMARKETS LIMITED_vs_HITT.pdf" # <-- change this

assert appeal_dir.exists(), f"Missing dir: {appeal_dir}"


PDF_PATH = appeal_dir / appeal_name  # <-- change this
assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"

def pdf_to_text(path: Path) -> str:
    out = []
    with path.open("rb") as f:
        reader = PyPDF2.PdfReader(f)
        for i, page in enumerate(reader.pages):
            txt = page.extract_text() or ""
            out.append(txt)
    return "\n".join(out)

doc_text = pdf_to_text(PDF_PATH)
print("PDF chars:", len(doc_text))
print("Preview:\n", doc_text[:1200])


PDF chars: 30440
Preview:
 A1/2002/0138  
Neutral Citation Number: [2002] EWCA Civ 1588   
IN THE SUPREME COURT OF JUDICATURE   
COURT OF APPEAL (CIVIL DIVISION)   
ON APPEAL FROM THE EMPLOYMENT APPEAL TRIBUNAL   
Royal Courts of Justice   
Strand  
London WC2  
Date: Friday, 18th October 2002   
B e f o r e :
LORD JUSTICE WARD   
LORD JUSTICE MUMMERY and   
LORD JUSTICE JONATHAN PARKER   
---------------------
SAINSBURYS SUPERMARKETS LIMITED
Appellant  
-v-
MR P J HITT  
Respondent  
------------------------
Computer Aided Transcript of the Palantype Notes of
Smith Bernal Reporting Limited
190 Fleet Street  London EC4A 2AG
Tel: 020 7421 4040  Fax: 020 7831 8838
(Official Shorthand Writers to the Court)
--------------------------
Mr J Galbraith-Marten (instructed by Group Legal Services, J Sainsbury Plc, London EC1) appeared  
on behalf of the Appellant.
The Respondent did not appear and was not represented.
------------------------
J U D G M E N T  
LORD JUSTICE WARD: I will ask Lord 

In [3]:
# === Moltie LLM verifier: single-document smoke test (TRIMMED) ===
# End-to-end: PDF -> paras(with stable para_id) -> AtomQuery -> prompt -> Ollama -> Verdict

from pathlib import Path
import sys
import PyPDF2

# -----------------------
# 0) Paths / imports
# -----------------------
PROJECT_ROOT = Path("/home/hello/Projects/Statements/code")  # adjust if needed
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from moltie.schemas.query_object import build_atom_from_y_path
from moltie.schemas.run_config import RunConfig
from moltie.llm.verifier_prompt import build_verifier_prompt
from moltie.llm.client import verify_with_ollama, LLMClientConfig

# -----------------------
# 1) Load one PDF -> text
# -----------------------
appeal_dir = Path("/home/hello/Projects/Statements/code/appeals/")
appeal_name = "SAINSBURYS SUPERMARKETS LIMITED_vs_HITT.pdf"  # <-- change this
PDF_PATH = appeal_dir / appeal_name

assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"

def pdf_to_text(path: Path) -> str:
    out = []
    with path.open("rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            out.append(page.extract_text() or "")
    return "\n".join(out)

doc_text = pdf_to_text(PDF_PATH)
print("PDF chars:", len(doc_text))
print("Preview:\n", doc_text[:800])

# -----------------------
# 2) Paragraph split + stable IDs (NO regex)
# -----------------------
def clean_para(s: str) -> str:
    # normalize whitespace without regex
    return " ".join((s or "").strip().split())

def to_paras(text: str, min_len: int = 40):
    paras = []
    buf = []
    n = 0

    for line in (text or "").splitlines():
        if line.strip():
            buf.append(line)
        else:
            if buf:
                p = clean_para("\n".join(buf))
                buf = []
                if len(p) >= min_len:
                    n += 1
                    paras.append({"para_id": f"p{n:05d}", "text": p})

    # flush tail
    if buf:
        p = clean_para("\n".join(buf))
        if len(p) >= min_len:
            n += 1
            paras.append({"para_id": f"p{n:05d}", "text": p})

    return paras

paras = to_paras(doc_text)
print("Paras:", len(paras))
assert paras, "No paragraphs extracted — PDF text extraction likely failed."

para_id_to_idx = {p["para_id"]: i for i, p in enumerate(paras)}

# -----------------------
# 3) AtomQuery (debug)
# -----------------------
Y_PATH = "/home/hello/Projects/Statements/output/Y_inferred.json"

# DO NOT overwrite original Y during smoke tests
Y_DEDUP_OUT = None  # disable dedup write-back

# comes from whatever step generates your evidence hits (debug stub here)
evidence_to_x = {
    "p00029": ["X1", "X3"],
    "p00030": ["X1"],
    "p00031": ["X4", "X1"],
}

atom, deduped_y = build_atom_from_y_path(
    atom_id="X_DEBUG",
    y_path=Y_PATH,
    evidence_to_x=evidence_to_x,
    dedup_out_path=Y_DEDUP_OUT,   # no file write
)

print("AtomQuery x_tests:", atom.x_tests)

# DO NOT print dedup path (because none written)
# print("Wrote deduped Y view to:", Y_DEDUP_OUT)

cfg = RunConfig(
    anchors_required=2,
    thresh_score=70,
    thresh_conf=75,
    y_path=Y_PATH,
    y_dedup_out=None,  # explicitly disable
)


# -----------------------
# 4) Evidence pack (deterministic, NO keyword heuristics)
#    Strategy:
#      - If evidence_to_x contains para_ids present in this doc, take those + neighbor window
#      - Else take a simple first-K slice (or mid-doc slice)
# -----------------------
K_DEBUG_PARAS = 12
NEIGHBOR_WINDOW = 3  # +/- around referenced para_ids

def select_debug_paras(paras, para_id_to_idx, preferred_para_ids):
    idxs = set()

    # include referenced paras + neighbors (when present)
    for pid in preferred_para_ids:
        i = para_id_to_idx.get(pid)
        if i is None:
            continue
        lo = max(0, i - NEIGHBOR_WINDOW)
        hi = min(len(paras) - 1, i + NEIGHBOR_WINDOW)
        idxs.update(range(lo, hi + 1))

    if idxs:
        chosen = [paras[i] for i in sorted(idxs)]
        # cap to K by taking a centered window around the median chosen index
        if len(chosen) > K_DEBUG_PARAS:
            mid = len(chosen) // 2
            half = K_DEBUG_PARAS // 2
            chosen = chosen[max(0, mid - half) : max(0, mid - half) + K_DEBUG_PARAS]
        return chosen, {"method": "evidence_window", "note": "para_id hits + neighbors"}

    # fallback: first K paras (or mid slice if doc is huge)
    if len(paras) <= K_DEBUG_PARAS:
        return paras, {"method": "first_k", "note": "doc shorter than K"}
    return paras[:K_DEBUG_PARAS], {"method": "first_k", "note": "no para_id hits; first-K fallback"}

preferred_para_ids = list(evidence_to_x.keys())
top_paras, retrieval_meta = select_debug_paras(paras, para_id_to_idx, preferred_para_ids)

evidence_pack = {
    "doc_id": PDF_PATH.stem,
    "doc_meta": {"source_path": str(PDF_PATH)},
    "paras": top_paras,
    "retrieval": {"method": retrieval_meta["method"], "note": retrieval_meta["note"], "score": None},
}

print("Evidence pack paras:", len(evidence_pack["paras"]))
print("Retrieval:", evidence_pack["retrieval"])
print("\n--- Evidence pack preview (para_id, snippet) ---")
for p in evidence_pack["paras"][:8]:
    print(p["para_id"], "|", p["text"][:180], "..." if len(p["text"]) > 180 else "")

# -----------------------
# 5) Build prompt + call Ollama
# -----------------------
prompt = build_verifier_prompt(atom=atom, evidence_pack=evidence_pack, cfg=cfg)

client_cfg = LLMClientConfig(
    model="mistral-small3.2:latest",
    ollama_url="http://localhost:11434/api/generate",
    timeout_s=180,
    temperature=0.0,
    num_predict=800,
    max_retries=2,
)

verdict = verify_with_ollama(prompt, client_cfg)

# -----------------------
# 6) Inspect result
# -----------------------
print("\n=== VERDICT ===")
print(verdict.to_dict())

print("\n=== ANCHORS (verbatim) ===")
for a in verdict.anchors:
    print(f"- {a.para_id}: {a.quote[:180]}{'...' if len(a.quote)>180 else ''}")
    print(f"  why: {a.why_it_matters}")


PDF chars: 30440
Preview:
 A1/2002/0138  
Neutral Citation Number: [2002] EWCA Civ 1588   
IN THE SUPREME COURT OF JUDICATURE   
COURT OF APPEAL (CIVIL DIVISION)   
ON APPEAL FROM THE EMPLOYMENT APPEAL TRIBUNAL   
Royal Courts of Justice   
Strand  
London WC2  
Date: Friday, 18th October 2002   
B e f o r e :
LORD JUSTICE WARD   
LORD JUSTICE MUMMERY and   
LORD JUSTICE JONATHAN PARKER   
---------------------
SAINSBURYS SUPERMARKETS LIMITED
Appellant  
-v-
MR P J HITT  
Respondent  
------------------------
Computer Aided Transcript of the Palantype Notes of
Smith Bernal Reporting Limited
190 Fleet Street  London EC4A 2AG
Tel: 020 7421 4040  Fax: 020 7831 8838
(Official Shorthand Writers to the Court)
--------------------------
Mr J Galbraith-Marten (instructed by Group Legal Services, J Sainsbury Plc, London EC1)
Paras: 1
AtomQuery x_tests: ['X1']
Evidence pack paras: 1
Retrieval: {'method': 'first_k', 'note': 'doc shorter than K', 'score': None}

--- Evidence pack preview (para_id,

### Moltie agent execution (single document)

This cell is the **minimal execution entry point** for Moltie.

It invokes the Moltie agent loop on:
- one document (identified by `doc_id`)
- pre-split paragraphs with stable `para_id`s
- a fully constructed `AtomQuery`
- runtime and LLM configuration

The agent loop evaluates the evidence and returns exactly one outcome:
- a **Verdict** if the atom is supported, or
- a **NegativeExit** if the agent cannot validate it.

This cell assumes all inputs are valid and integrated.
It represents the **canonical way Moltie is run**.


# Moltie — Single‑Document Flow

This file explains **what runs and in what order** when Moltie evaluates **one document against one atom**.

---

## Entry point

You call **one function only**:

```python
res = run_agent_on_one_doc(doc_id, paras, atom, run_cfg, llm_cfg)
```

The result is **exactly one** of:
- `Verdict` → accepted evidence‑anchored result  
- `NegativeExit` → explicit, audited stop condition

---

## What happens inside

1) **Retrieve (recall only)**  
`retrieve_windowed_evidence`  
Selects a small window of paragraphs and tracks coverage.

2) **Ask (build prompt)**  
`build_verifier_prompt`  
Compiles AtomQuery + evidence + strict Verdict JSON contract.

3) **Verify (LLM single‑shot)**  
`verify_with_ollama`  
Calls the LLM once and returns a schema‑valid `Verdict` (repair/retry if needed).

4) **Truth gate**  
Agent checks every anchor:
- `para_id` exists  
- quote is verbatim  
Failure ⇒ reject.

5) **Score progress**  
Agent computes a quality metric to detect improvement or plateau.

6) **Decide**  
- Accept ⇒ return `Verdict`  
- Plateau / exhausted ⇒ return `NegativeExit`  
- Else ⇒ refine query and loop

---

## Key rule

**The agent controls the loop.  
The LLM answers one question.**


In [4]:
# ==========================================================
# MOLTIE SINGLE-DOC RUN HARNESS (REAL LOOP, NO FAKE evidence_to_x)
# ==========================================================

import sys, importlib, json
from pathlib import Path
from pprint import pprint

# -------------------------
# 0) Explicit repo anchors
# -------------------------
REPO_ROOT = Path("/home/hello/Projects/Statements").resolve()
CODE_ROOT = (REPO_ROOT / "code").resolve()
assert REPO_ROOT.exists(), f"Missing repo root: {REPO_ROOT}"
assert CODE_ROOT.exists(), f"Missing code root: {CODE_ROOT}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# -------------------------
# 1) Import modules (then reload in dependency order)
# -------------------------
import moltie.schemas.run_config as rc_mod
import moltie.llm.verifier_prompt as vp_mod
import moltie.llm.client as client_mod
import moltie.schemas.query_object as qo_mod
import moltie.agent.loop as loop_mod

importlib.reload(rc_mod)      # config first
importlib.reload(vp_mod)      # prompt
importlib.reload(client_mod)  # client boundary
importlib.reload(qo_mod)      # AtomQuery helpers
importlib.reload(loop_mod)    # loop last

RunConfig = rc_mod.RunConfig
AtomQuery = qo_mod.AtomQuery
run_agent_on_one_doc = loop_mod.run_agent_on_one_doc

print("[moltie.debug] repo:", REPO_ROOT)
print("[moltie.debug] code:", CODE_ROOT)
print("[moltie.debug] loop file:", Path(loop_mod.__file__).resolve())

# -------------------------
# 2) REQUIRED INPUTS (must already exist in notebook)
# -------------------------
# You must have these in your notebook already from your parsing step:
#   PDF_PATH: Path to the PDF (used only for doc_id)
#   paras:    List[{"para_id": "...", "text": "..."}]
#   client_cfg: LLMClientConfig instance
#
# If any are missing, fail fast with a crisp message.
for name in ("PDF_PATH", "paras", "client_cfg"):
    if name not in globals():
        raise RuntimeError(f"Missing required notebook variable: {name}")

assert isinstance(PDF_PATH, Path), "PDF_PATH must be a pathlib.Path"
assert PDF_PATH.exists(), f"Missing PDF file: {PDF_PATH}"
assert isinstance(paras, list) and len(paras) > 0, "paras must be non-empty list"
assert isinstance(paras[0], dict) and "para_id" in paras[0] and "text" in paras[0], "paras must be list of dicts with para_id/text"

DOC_ID = PDF_PATH.stem

print("[moltie.debug] doc_id:", DOC_ID)
print("[moltie.debug] paras:", len(paras), "sample:", paras[0]["para_id"], "len:", len(paras[0]["text"] or ""))

# -------------------------
# 3) Load Y JSON (real file)
# -------------------------
Y_PATH = REPO_ROOT / "output" / "Y_inferred.json"
assert Y_PATH.exists(), f"Missing Y file: {Y_PATH}"
y_json = json.loads(Y_PATH.read_text(encoding="utf-8"))

x_tests_map = (y_json.get("x_tests") or {})
assert isinstance(x_tests_map, dict) and len(x_tests_map) > 0, "Y JSON has no x_tests"

print("[moltie.debug] Y file:", Y_PATH)
print("[moltie.debug] Y x_tests:", len(x_tests_map))

# -------------------------
# 4) Run loop for each X test in Y (real AtomQuery construction + real merging logic)
#    Note: you can run this loop for a subset of X tests by slicing the sorted keys of x_tests_map
# -------------------------

results = []

for x_key in sorted(x_tests_map.keys()):
    print("\n==================================================")
    print("Running atom:", x_key, "->", x_tests_map[x_key].get("name"))
    print("==================================================")

    merged = qo_mod.merge_indicators_and_excludes(y_json, [x_key])

    atom = AtomQuery(
        atom_id=x_key,
        x_tests=[x_key],
        proposition=x_tests_map[x_key].get("name", x_key),
        positive_indicators=merged["positive_indicators"],
        excludes=merged["excludes"],
        keyword_seeds=merged["positive_indicators"],
        expansion_terms=[],
    )

    cfg2 = RunConfig.from_dict({
        "debug": False,
        "harvest_mode": False,
        "max_iters": 1,
        "window_size": 12,
        "stride": 6,
        "top_windows": 3,
        "k_chunks_per_doc": 12,
        "anchors_required": 1,
        "min_hits": 1,
        "thresh_score": 1,
        "thresh_conf": 1,
        "plateau_p": 2,
        "eps_improve": 0,
    })

    res = run_agent_on_one_doc(
        DOC_ID,
        paras,
        atom,
        cfg2,
        client_cfg,
    )

    results.append((x_key, res))

    print("\nRESULT for", x_key)
    if res.verdict:
        v = res.verdict.to_dict()
        print("relevant=", v.get("relevant"),
              "score=", v.get("precedent_score"),
              "conf=", v.get("confidence"),
              "anchors=", len(v.get("anchors") or []))
        pprint(v.get("anchors"))
    else:
        print("NEGATIVE:", res.negative_exit.reason if res.negative_exit else None)

# -------------------------
# 5) Output (compact + useful)
# -------------------------
print("\n================ RESULT ================")
if res.verdict:
    v = res.verdict.to_dict()
    print("VERDICT: relevant=", v.get("relevant"),
          "score=", v.get("precedent_score"),
          "conf=", v.get("confidence"),
          "anchors=", len(v.get("anchors") or []),
          "matched_X=", v.get("matched_X"))
    print("\nanchors:")
    pprint(v.get("anchors"))
else:
    print("NEGATIVE_EXIT:")
    pprint(res.negative_exit.to_dict() if res.negative_exit else None)

print("\n================ TRACE (last 3 entries) ================")
pprint(res.trace[-3:])

print("\n================ TRACE SUMMARY ================")
print("iters:", res.iters, "trace_len:", len(res.trace))


[moltie.debug] repo: /home/hello/Projects/Statements
[moltie.debug] code: /home/hello/Projects/Statements/code
[moltie.debug] loop file: /home/hello/Projects/Statements/code/moltie/agent/loop.py
[moltie.debug] doc_id: SAINSBURYS SUPERMARKETS LIMITED_vs_HITT
[moltie.debug] paras: 1 sample: p00001 len: 29390
[moltie.debug] Y file: /home/hello/Projects/Statements/output/Y_inferred.json
[moltie.debug] Y x_tests: 6

Running atom: X1 -> Inadequate Investigation Scope
[moltie.loop] start doc_id='SAINSBURYS SUPERMARKETS LIMITED_vs_HITT' atom_id='X1' n_paras=18

[moltie.client] ===== Attempt A =====
[moltie.client] attempt: 0
[moltie.client] prompt_hash: ab49201c92
[moltie.client] prompt_head:
 TOP RULES (ABSOLUTE):
- atom_id MUST be 'X1' exactly.
- doc_id MUST be 'SAINSBURYS SUPERMARKETS LIMITED_vs_HITT' exactly.
- matched_X MUST be a subset of atom.x_tests=['X1']. If relevant=true, matched_X MUST include 'X1'.
- NEVER output more than 3 matched_X items. If unsure, matched_X=[].
- If you canno

In [5]:
import moltie.llm.verifier_prompt as vp_mod
from pathlib import Path

print("verifier_prompt file:", Path(vp_mod.__file__).resolve())
print("schema snippet contains atom.x_tests rule?:",
      "subset of Input JSON atom.x_tests" in vp_mod._VERDICT_SCHEMA_TEXT)

# also print the exact tail where your rules live
print("\n--- schema tail ---\n", vp_mod._VERDICT_SCHEMA_TEXT[-500:])


verifier_prompt file: /home/hello/Projects/Statements/code/moltie/llm/verifier_prompt.py
schema snippet contains atom.x_tests rule?: True

--- schema tail ---
  you MUST copy-paste text EXACTLY from the paragraph.
- Do NOT paraphrase. Do NOT retype from memory. Do NOT change apostrophes/quotes.
- The quote MUST appear as a contiguous substring in that paragraph.
- If you cannot comply, set relevant=false and anchors=[].
- matched_X MUST be a subset of Input JSON atom.x_tests.
- If relevant=true, matched_X MUST contain atom.atom_id.
- If unsure, use [].
- atom_id MUST equal Input JSON atom.atom_id exactly.
- doc_id MUST equal Input JSON doc_id exactly.

